# It's the Student, Not the Thesis — Presentation Results Pipeline

Colab-ready notebook: clone from GitHub, run Qwen 7B on GPU, generate all six presentation results (diagnostician validation, committee assembly, side-by-side feedback, development plan, quantitative comparison). Update!


In [ ]:
# Cell 1: Colab detect + clone from GitHub
import os, sys
if "google.colab" in sys.modules:
    get_ipython().system("git clone https://github.com/anfelipecb/AI-Agents-Final-Project.git")
    os.chdir("AI-Agents-Final-Project")
    if os.path.exists("final-project"):
        os.chdir("final-project")

In [ ]:
# Cell 2: Install deps
get_ipython().system("pip install -q httpx beautifulsoup4 sentence-transformers transformers accelerate bitsandbytes numpy scikit-learn matplotlib seaborn openai")
# If project has pyproject.toml/setup.py: pip install -e .
try:
    get_ipython().system("pip install -e .")
except Exception:
    pass

## Cell 3: Paths and GPU check

In [ ]:
import sys
from pathlib import Path

# Add project src to path (adjust if repo structure differs)
for base in [Path.cwd(), Path.cwd().parent, Path.cwd() / "final-project"]:
    src = base / "src"
    if (src / "developing_the_researcher").exists():
        sys.path.insert(0, str(src))
        break

import torch
assert torch.cuda.is_available(), "Enable GPU: Runtime -> Change runtime type -> T4/A100"
print("GPU available:", torch.cuda.get_device_name(0))

## Load corpus and models (Qwen 7B only)

In [ ]:
from developing_the_researcher.config import DOCS_VALIDATION_OUTPUTS
from developing_the_researcher.data import CorpusLoader
from developing_the_researcher.models import CommitteeLoader, EmbeddingLoader, get_generate_fn

corpus = CorpusLoader()
theses = corpus.load()
embed_loader = EmbeddingLoader()
committee = CommitteeLoader()  # Qwen 7B
print(f"Loaded {len(theses)} theses")

## Result 2: Diagnostician validation

Sample theses by cluster, run 6-dim diagnostician, produce radar charts.

In [ ]:
from developing_the_researcher.analysis.diagnostician_validation import run_validation

# 7B only. Use OpenAI if USE_OPENAI=1 for faster validation.
diag_results = run_validation(
    n_theses=10,
    models=["Qwen/Qwen2.5-7B-Instruct"],
    out_figures_dir=DOCS_VALIDATION_OUTPUTS,
)
print("Result 2: Radar charts saved to", DOCS_VALIDATION_OUTPUTS)

## Result 3: Committee assembly demo

Different profiles trigger different committees. Plot thesis → profile → committee for 2–3 theses.

In [ ]:
from developing_the_researcher.analysis.committee_assembly_figure import plot_committee_assembly_demo
from developing_the_researcher.analysis.diagnostician_validation import sample_theses_by_cluster

sampled = sample_theses_by_cluster(theses, embed_loader, n_total=3, n_clusters=6)
out_r3 = plot_committee_assembly_demo(sampled, committee, DOCS_VALIDATION_OUTPUTS / "committee_assembly_demo.png")
print("Result 3: Saved to", out_r3)

## Result 4: Side-by-side feedback (C1 vs C3)

Run C1 and C3 for 2–3 theses; plot side-by-side and deliberation excerpts.

In [ ]:
from developing_the_researcher.analysis.feedback_comparison import run_side_by_side_comparison

paths_r4 = run_side_by_side_comparison(sampled, committee, embed_loader, DOCS_VALIDATION_OUTPUTS)
print("Result 4: Saved", len(paths_r4), "figures")

## Result 5: Development plan example

Run C3 for 1 thesis, get development plan, plot gap_map, exercises, trajectory.

In [ ]:
from developing_the_researcher.analysis.development_plan_figure import plot_development_plan_example
from developing_the_researcher.models.development_plan import development_plan
from developing_the_researcher.models.diagnostician import diagnose_competencies

t = sampled[0]
text = f"{(t.get('title') or '')}. {(t.get('abstract') or '')}"[:800]
def _gen(p, s, m): return committee.generate(p, s, m)
profile = diagnose_competencies(text, _gen)
agents = committee.assemble_committee(profile)
consolidated = committee.committee_deliberation(text, agents)
plan = development_plan(profile, consolidated, _gen)
out_r5 = plot_development_plan_example(plan, t.get("title", "Example"), DOCS_VALIDATION_OUTPUTS / "development_plan_example.png")
print("Result 5: Saved to", out_r5)

## Result 6: Quantitative experiment (3×3×5)

Full experiment: 3 students × 3 conditions × 5 theses. Saves bar charts (mechanical reliance, plan quality, trust, feedback specificity).

In [ ]:
# Quick mode (2×3×2) for testing; set full_experiment=True for 3×3×5 (~2-4 hrs on A100)
from developing_the_researcher.pipeline import run_pilot

pilot_results = run_pilot(n_per_condition=2, save_figures=True, full_experiment=False)
print("Result 6: Pilot complete. For full experiment, set full_experiment=True.")

## Output summary

Paths to all saved figures in docs/validation_outputs/

In [ ]:
import os
for f in sorted(DOCS_VALIDATION_OUTPUTS.glob("*")):
    print(f)

In [ ]:
# Zip and download results (Colab only — run before runtime disconnects)
from pathlib import Path
import shutil

output_dir = Path("docs/validation_outputs")
data_dir = Path("data")
results_zip = "colab_results.zip"

with shutil.ZipFile(results_zip, "w", shutil.ZIP_DEFLATED) as zf:
    if output_dir.exists():
        for f in output_dir.glob("*"):
            zf.write(f, f"validation_outputs/{f.name}")
    for name in ["pilot_results.json", "diagnostician_validation.json"]:
        p = data_dir / name
        if p.exists():
            zf.write(p, f"data/{name}")

if "google.colab" in __import__("sys").modules:
    from google.colab import files
    files.download(results_zip)
    print("Download started. Check your browser downloads.")
else:
    print(f"Results zipped to {results_zip} (run in Colab to download)")

---

Pilot complete. See `report.md` and `README.md` for method and qualitative interpretation.